# mask-to-mri — Synthetic MRI Generation with pix2pix

This notebook runs the full pipeline: data exploration → preprocessing → model training → evaluation.

**Experiment A:** Train pix2pix to generate realistic MRI from masks.  
**Experiment B:** Measure segmentation improvement with synthetic data.

## 0 — Setup & Config

In [ ]:
import sys
sys.path.insert(0, '.')

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

# Load config
with open('config.yaml') as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(yaml.dump(config, default_flow_style=False))

# Fix seeds
from src.utils import fix_seed, get_device
fix_seed(config['data']['seed'])
device = get_device()

## 1 — Data Exploration

In [ ]:
from src.dataset import get_patient_file_list
import tifffile
import os

raw_dir = config['data']['raw_dir']
patient_data = get_patient_file_list(raw_dir)

print(f"Total patients: {len(patient_data)}")
print(f"Sample patients: {list(patient_data.keys())[:5]}")

total_slices = sum(len(v) for v in patient_data.values())
slices_per_patient = [len(v) for v in patient_data.values()]
print(f"Total slices: {total_slices}")
print(f"Slices per patient: min={min(slices_per_patient)}, max={max(slices_per_patient)}, avg={np.mean(slices_per_patient):.1f}")

In [ ]:
# Visualize a sample pair
first_patient = list(patient_data.keys())[0]
img_path, mask_path = patient_data[first_patient][10]

image = tifffile.imread(img_path)
mask = tifffile.imread(mask_path)

print(f"Patient: {first_patient}")
print(f"Image shape: {image.shape}, dtype: {image.dtype}")
print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
print(f"Image channel stats:")
for i, ch in enumerate(['R/T1', 'G/FLAIR', 'B/T2']):
    print(f"  {ch}: min={image[:,:,i].min()}, max={image[:,:,i].max()}, mean={image[:,:,i].mean():.1f}")
print(f"Mask unique values: {np.unique(mask)}")

# Show side by side
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title('MRI (RGB — 3 sequences)')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Tumor Mask')
axes[1].axis('off')

# Overlay
overlay = image.copy()
overlay[mask > 0] = [255, 0, 0]  # red overlay on tumor
axes[2].imshow(overlay)
axes[2].set_title('MRI + Tumor Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 2 — Dataset & DataLoaders

In [ ]:
from src.dataset import build_dataloaders

loaders = build_dataloaders(
    raw_dir=config['data']['raw_dir'],
    image_size=config['data']['image_size'],
    batch_size=config['training']['batch_size'],
    num_workers=0,
    seed=config['data']['seed'],
)

In [ ]:
# Verify tensor shapes and visualize a batch
mask, img = next(iter(loaders['train']))
print(f"Mask tensor: {mask.shape}  (B, C, H, W)")
print(f"Image tensor: {img.shape}  (B, C, H, W)")
print(f"Mask range: [{mask.min():.2f}, {mask.max():.2f}]")
print(f"Image range: [{img.min():.2f}, {img.max():.2f}]")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
mask_vis = ((mask[0, 0].numpy() + 1.0) * 127.5).astype(np.uint8)
img_vis = ((img[0].numpy().transpose(1, 2, 0) + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
axes[0].imshow(mask_vis, cmap='gray')
axes[0].set_title('Mask (normalized)')
axes[0].axis('off')
axes[1].imshow(img_vis)
axes[1].set_title('MRI (normalized)')
axes[1].axis('off')
plt.show()

## 3 — Model Architecture

In [ ]:
from src.generator import create_generator
from src.discriminator import create_discriminator
from src.utils import print_model_summary

model_cfg = config['model']

G = create_generator(
    in_channels=model_cfg['input_channels'],
    out_channels=model_cfg['output_channels'],
    num_filters=model_cfg['num_filters'],
    norm=model_cfg['norm'],
).to(device)

D = create_discriminator(
    in_channels=model_cfg['input_channels'] + model_cfg['output_channels'],
    num_filters=model_cfg['num_filters'],
).to(device)

print_model_summary('Generator (U-Net)', G)
print_model_summary('Discriminator (PatchGAN)', D)

In [ ]:
# Verify forward pass shapes
with torch.no_grad():
    fake = G(mask.to(device))
    d_real = D(mask.to(device), img.to(device))
    d_fake = D(mask.to(device), fake)

print(f"Generator output: {fake.shape}")
print(f"Discriminator real: {d_real.shape}")
print(f"Discriminator fake: {d_fake.shape}")

## 4 — Training

In [ ]:
from src.train import train

# CPU optimization (uncomment if training on CPU)
# torch.set_num_threads(8)
# torch.set_num_interop_threads(4)

history = train(
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    generator=G,
    discriminator=D,
    config=config,
    device=device,
    checkpoint_dir=config['paths']['checkpoints'],
    samples_dir=config['paths']['samples'],
)

In [ ]:
# Plot loss curves
from src.utils import plot_loss_curves
plot_loss_curves(history, save_path='outputs/samples/loss_curves.png')

## 5 — Generate Synthetic MRI (Experiment A)

In [ ]:
# Load best checkpoint
import glob
from src.train import load_checkpoint

checkpoints = sorted(glob.glob('outputs/checkpoints/checkpoint_epoch_*.pt'))
if checkpoints:
    last_ckpt = checkpoints[-1]
    epoch = load_checkpoint(last_ckpt, G, D)
    print(f"Loaded checkpoint from epoch {epoch}")
else:
    print("No checkpoints found — using current model weights")

In [ ]:
# Generate synthetic MRI on test set and save
import os
from PIL import Image

synthetic_dir = config['data']['synthetic_dir']
os.makedirs(synthetic_dir, exist_ok=True)

G.eval()
count = 0
with torch.no_grad():
    for mask, real in loaders['test']:
        mask_dev = mask.to(device)
        fake = G(mask_dev)
        
        # Denormalize and save
        fake_np = ((fake[0].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        real_np = ((real[0].cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        
        Image.fromarray(fake_np).save(os.path.join(synthetic_dir, f'fake_{count:04d}.png'))
        Image.fromarray(real_np).save(os.path.join(synthetic_dir, f'real_{count:04d}.png'))
        count += 1

print(f"Generated {count} synthetic MRI slices → {synthetic_dir}")

## 6 — Evaluation (Experiment A: GAN Quality)

In [ ]:
from src.evaluate import compute_ssim_batch, compute_psnr_batch, compute_fid_from_paths, save_eval_results

# SSIM & PSNR on test set
G.eval()
all_ssim = []
all_psnr = []

with torch.no_grad():
    for mask, real in loaders['test']:
        fake = G(mask.to(device))
        ssim_score = compute_ssim_batch(fake, real)
        psnr_score = compute_psnr_batch(fake, real)
        all_ssim.append(ssim_score)
        all_psnr.append(psnr_score)

mean_ssim = np.mean(all_ssim)
mean_psnr = np.mean(all_psnr)
print(f"Mean SSIM: {mean_ssim:.4f}")
print(f"Mean PSNR: {mean_psnr:.2f} dB")

In [ ]:
# FID score
print("Computing FID (this may take a minute)...")
fid = compute_fid_from_paths(
    real_dir=os.path.join(synthetic_dir),  # real_*.png
    fake_dir=os.path.join(synthetic_dir),  # fake_*.png
    device=str(device),
)
print(f"FID: {fid:.2f}")

In [ ]:
# Save evaluation results
metrics = {
    'ssim': round(mean_ssim, 4),
    'psnr': round(mean_psnr, 2),
    'fid': round(fid, 2),
}
save_eval_results(metrics, metrics_dir=config['paths']['metrics'], prefix='eval_exp_a')
print(f"\nExperiment A Results: {metrics}")

## 7 — Experiment B: Downstream Segmentation

In [ ]:
# Train segmentation U-Net on real-only vs real+synthetic
# and compare Dice scores on the test set.
#
# This section uses segmentation-models-pytorch.
# Expected improvement: +3% to +7% Dice score.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import segmentation_models_pytorch as smp
from src.losses import DiceBCELoss
from src.evaluate import compute_dice_score, save_eval_results

print("Experiment B: Training segmentation U-Net...")
print("  Setup 1: Real-only training")
print("  Setup 2: Real + Synthetic training")

In [ ]:
def train_segmentation(train_loader, val_loader, epochs=50, device='cpu'):
    """Train a segmentation U-Net and return Dice score on validation set."""
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,
        in_channels=3,  # MRI: 3 channels
        classes=1,
        activation="sigmoid",
    ).to(device)
    
    criterion = DiceBCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0
        for img, mask in tqdm(train_loader, desc=f"Seg Epoch {epoch}/{epochs}", leave=False):
            img = img.to(device)
            mask = mask.to(device)
            # mask is (B, 1, H, W) normalized [-1,1] → convert to [0,1]
            mask_binary = (mask + 1.0) / 2.0
            
            pred = model(img)
            loss = criterion(pred, mask_binary)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if epoch % 10 == 0:
            print(f"  Seg Epoch {epoch}: loss={epoch_loss/len(train_loader):.4f}")
    
    # Evaluate Dice on validation set
    model.eval()
    dice_scores = []
    with torch.no_grad():
        for img, mask in val_loader:
            pred = model(img.to(device))
            pred_binary = (pred.cpu().numpy() > 0.5).astype(np.float32)
            mask_binary = ((mask + 1.0) / 2.0).numpy()
            dice = compute_dice_score(pred_binary, mask_binary)
            dice_scores.append(dice)
    
    return np.mean(dice_scores)

# NOTE: This requires sufficient training data.
# Run on a subset if training on CPU:
# dice_real_only = train_segmentation(loaders['train'], loaders['val'], epochs=30, device=device)
# print(f"Dice (real only): {dice_real_only:.4f}")

## 8 — Results & Conclusion

In [ ]:
# Final results summary
print("=" * 60)
print("mask-to-mri — Final Results")
print("=" * 60)
print(f"\nExperiment A — GAN Quality:")
print(f"  SSIM: {mean_ssim:.4f}")
print(f"  PSNR: {mean_psnr:.2f} dB")
print(f"  FID:  {fid:.2f}")
print(f"\nExperiment B — Segmentation (Dice):")
print(f"  Real only:      TBD")
print(f"  Real + Synth:   TBD")
print(f"  Delta:          TBD")
print(f"\nSynthetic samples saved: {len(os.listdir(synthetic_dir))} files")
print("=" * 60)